# Kubeflow Pipelines — Endpoint Tests

Tests all KFP REST API endpoints on the DGX Spark cluster.

**Prerequisites:** KFP deployed + SSH tunnel open on ports 8080 and 8890.

```sh
ssh -L 8080:localhost:8080 -L 8890:localhost:8890 -L 8888:localhost:8888 <user>@spark-79b7.local
```

In [ ]:
import requests, json

API = "http://localhost:8890/apis/v2beta1"
UI  = "http://localhost:8080"

def check(label, r):
    status = "\u2705" if r.ok else "\u274c"
    print(f"{status} {label}: HTTP {r.status_code}")
    if not r.ok:
        print(f"   {r.text[:300]}")
    return r

## Health

In [ ]:
r = check("API healthz", requests.get(f"{API}/healthz"))
if r.ok:
    print(json.dumps(r.json(), indent=2))

r2 = requests.get(UI)
print(f"{'\u2705' if r2.ok else '\u274c'} KFP UI: HTTP {r2.status_code} ({len(r2.text)} bytes)")

## Pipelines

In [ ]:
r = check("List pipelines", requests.get(f"{API}/pipelines"))
if r.ok:
    pipelines = r.json().get("pipelines", [])
    print(f"  {len(pipelines)} pipeline(s)")
    for p in pipelines:
        print(f"  - {p.get('display_name', '?')} ({p.get('pipeline_id', '?')})")

## Experiments

In [ ]:
r = check("List experiments", requests.get(f"{API}/experiments"))
if r.ok:
    exps = r.json().get("experiments", [])
    print(f"  {len(exps)} experiment(s)")
    for e in exps:
        print(f"  - {e.get('display_name', '?')} ({e.get('experiment_id', '?')})")

## Runs

In [ ]:
r = check("List runs", requests.get(f"{API}/runs"))
if r.ok:
    runs = r.json().get("runs", [])
    print(f"  {len(runs)} run(s)")
    for run in runs[:5]:
        print(f"  - {run.get('display_name', '?')} [{run.get('state', '?')}]")

## Summary

In [ ]:
results = [
    ("API healthz",      requests.get(f"{API}/healthz")),
    ("KFP UI",           requests.get(UI)),
    ("List pipelines",   requests.get(f"{API}/pipelines")),
    ("List experiments", requests.get(f"{API}/experiments")),
    ("List runs",        requests.get(f"{API}/runs")),
]
print("\n=== Summary ===")
for label, r in results:
    print(f"{'\u2705' if r.ok else '\u274c'} {label}: HTTP {r.status_code}")
failed = sum(1 for _, r in results if not r.ok)
print(f"\n{len(results) - failed}/{len(results)} passed")